In [92]:
import numpy as np
import pyvista as pv
pv.set_jupyter_backend('trame')

In [100]:
def unit_vec_from_angles(theta, phi):
    return np.array([
        np.sin(theta) * np.cos(phi), 
        np.sin(theta) * np.sin(phi), 
        np.cos(theta)
    ])

def cross_product_matrix(x1, x2, x3):
    return np.array([
        [0*x1, -x3, x2],
        [x3, 0*x2, -x1],
        [-x2, x1, 0*x3]])


def rotation_matrix_about_axis(alpha, x1, x2, x3):
    cp_mat = cross_product_matrix(x1, x2, x3)  # Shape: (3, 3, n)
    cp_mat_sq = np.einsum('ij...,jk...->ik...', cp_mat, cp_mat)
    t2 = np.sin(alpha) * cp_mat
    t3 = (1 - np.cos(alpha)) * cp_mat_sq
    return np.eye(3) + t2 + t3


def rotation_matrix_about_sphere_angles(alpha, theta, phi):
    unit_vec = unit_vec_from_angles(theta, phi)
    return rotation_matrix_about_axis(alpha, *unit_vec)


def theta_hat(theta, phi):
    theta_x = np.cos(theta) * np.cos(phi)
    theta_y = np.cos(theta) * np.sin(phi)
    theta_z = -np.sin(theta)
    return np.array([theta_x, theta_y, theta_z])

def phi_hat(theta, phi):
    phi_x = -np.sin(phi)
    phi_y = np.cos(phi)
    phi_z = 0
    return np.array([phi_x, phi_y, phi_z])

def get_theta_phi_from_unit_vec(x1, x2, x3):
    theta = np.arccos(x3)
    phi = np.arctan2(x2, x1)
    return theta, phi

def get_spherical_basis_from_unit_vec(x1, x2, x3):
    theta, phi = get_theta_phi_from_unit_vec(x1, x2, x3)
    return theta_hat(theta, phi), phi_hat(theta, phi)

In [183]:
pvplt = pv.Plotter()

unit_sphere = pv.Sphere(radius=1.0)
pvplt.add_mesh(unit_sphere, color='grey', style='wireframe', opacity=0.9, show_edges=False)
pvplt.enable_hidden_line_removal(all_renderers=False)

origin = np.array([0, 0, 0])
_obs_lens_points = np.array(
    [[1.2, 0, 0],
     [0, 1.2, 0.5]], dtype=float
)
obs_lens_points = _obs_lens_points.copy()
lensing_pln_normal = np.cross(*obs_lens_points)
obs_ray = pv.Line(origin, obs_lens_points[0])
lens_ray = pv.Line(origin, obs_lens_points[1])
pvplt.add_mesh(obs_ray, color='dodgerblue', line_width=3, render_lines_as_tubes=False)
pvplt.add_mesh(lens_ray, color='black', line_width=3, render_lines_as_tubes=False)

theta = np.pi / 4
rot_mat = rotation_matrix_about_axis(theta, *lensing_pln_normal)
image_obs = rot_mat @ obs_lens_points[0]
image_obs /= np.linalg.norm(image_obs)

image_pos = pv.Sphere(radius=0.1, center=image_obs)
pvplt.add_mesh(image_pos, color='orange')
lensing_pln = pv.Plane(center=[0, 0, 0], direction=lensing_pln_normal, i_size=2, j_size=2)
pvplt.add_mesh(lensing_pln, color='blue', opacity=0.7, show_edges=False)

# Compute theta, phi hats
theta_h, phi_h = get_spherical_basis_from_unit_vec(*obs_lens_points[0])
theta_h_rot = rot_mat @ theta_h
phi_h_rot = rot_mat @ phi_h
theta_prime_h, phi_prime_h = get_spherical_basis_from_unit_vec(*image_obs)

arrow_list = []
for centre, direction, color in [
    (obs_lens_points[0], theta_h, 'grey'),
    (obs_lens_points[0], phi_h, 'grey'),
    (image_obs, theta_prime_h, 'grey'),
    (image_obs, phi_prime_h, 'grey'),
    (image_obs, theta_h_rot, 'red'),
    (image_obs, phi_h_rot, 'red'),
]:
    arrow = pv.Arrow(centre, direction)
    pvplt.add_mesh(arrow, color=color, show_edges=False)
    arrow_list.append(arrow)

def update_objects():
    global obs_lens_points
    global lensing_pln_normal
    global lensing_pln
    global theta
    global image_pos
    global arrow_list

    _lensing_pln = pv.Plane(center=[0, 0, 0], direction=lensing_pln_normal, i_size=2, j_size=2)
    lensing_pln.copy_from(_lensing_pln)
    rot_mat = rotation_matrix_about_axis(theta, *lensing_pln_normal)
    image_obs = rot_mat @ obs_lens_points[0]
    image_obs /= np.linalg.norm(image_obs)
    _image_pos = pv.Sphere(radius=0.1, center=image_obs)
    image_pos.copy_from(_image_pos)

    # Compute theta, phi hats
    # vectors.mapper.center = obs_lens_points[0]
    theta_h, phi_h = get_spherical_basis_from_unit_vec(*obs_lens_points[0])
    theta_h_rot = rot_mat @ theta_h
    phi_h_rot = rot_mat @ phi_h
    theta_prime_h, phi_prime_h = get_spherical_basis_from_unit_vec(*image_obs)

    for arrow, (centre, direction) in zip(arrow_list, [
        (obs_lens_points[0], theta_h),
        (obs_lens_points[0], phi_h),
        (image_obs, theta_prime_h),
        (image_obs, phi_prime_h),
        (image_obs, theta_h_rot),
        (image_obs, phi_h_rot),
    ]):
        _arrow = pv.Arrow(centre, direction, scale=0.2)
        arrow.copy_from(_arrow)

def callback(point, i):
    global obs_lens_points
    global lensing_pln
    global lensing_pln_normal

    _obs_lens_points[i] = point
    obs_lens_points = _obs_lens_points.copy()
    norms = np.linalg.norm(_obs_lens_points, axis=1)
    obs_lens_points /= norms[:, None]
    lensing_pln_normal = np.cross(*obs_lens_points)
    lensing_pln_normal /= np.linalg.norm(lensing_pln_normal)

    _obs_ray = pv.Line(origin, _obs_lens_points[0])
    _lens_ray = pv.Line(origin, _obs_lens_points[1])
    obs_ray.copy_from(_obs_ray)
    lens_ray.copy_from(_lens_ray)
    update_objects()

def update_angle(val):
    global theta
    theta = val
    update_objects()

pvplt.add_sphere_widget(callback, center=obs_lens_points, color=['dodgerblue', 'black'], radius=0.07)
pvplt.add_slider_widget(
    callback=update_angle,
    rng=[0, 2 * np.pi],
    value=1,
    title='theta',
    pointa=(0.025, 0.5),
    pointb=(0.25, 0.5),
    style='modern',
)
pvplt.show(interactive_update=True)

Widget(value='<iframe src="http://localhost:65489/index.html?ui=P_0x49f626850_118&reconnect=auto" class="pyvis…

In [ ]:
lensing_pln.

PolyData (0x366ace160)
  N Cells:    100
  N Points:   121
  N Strips:   0
  X Bounds:   -1.000e+00, 1.000e+00
  Y Bounds:   -1.000e+00, 1.000e+00
  Z Bounds:   0.000e+00, 0.000e+00
  N Arrays:   2

In [62]:
lensing_pln.point_data

pyvista DataSetAttributes
Association     : POINT
Active Scalars  : None
Active Vectors  : None
Active Texture  : TextureCoordinates
Active Normals  : Normals
Contains arrays :
    Normals                 float32    (121, 3)             NORMALS
    TextureCoordinates      float32    (121, 2)             TCOORDS

PolyData,Information
N Cells,43
N Points,101
N Strips,0
X Bounds,"-1.110e-17, 1.000e+00"
Y Bounds,"-1.000e-01, 1.000e-01"
Z Bounds,"-1.000e-01, 1.000e-01"
N Arrays,0


In [155]:
vectors.mapper.dataset

PolyData (0x499c8b5e0)
  N Cells:    60
  N Points:   124
  N Strips:   0
  X Bounds:   1.238e-01, 1.030e+00
  Y Bounds:   -3.000e-02, 8.224e-01
  Z Bounds:   -3.000e-01, 6.793e-01
  N Arrays:   2

In [151]:
vec_data

PolyData (0x494584880)
  N Cells:    4
  N Points:   4
  N Strips:   0
  X Bounds:   3.827e-01, 1.000e+00
  Y Bounds:   0.000e+00, 6.533e-01
  Z Bounds:   0.000e+00, 6.533e-01
  N Arrays:   2

In [146]:
rot_vectors.mapper.dataset

PolyData (0x491078580)
  N Cells:    30
  N Points:   62
  N Strips:   0
  X Bounds:   1.706e-01, 5.948e-01
  Y Bounds:   6.443e-01, 8.654e-01
  Z Bounds:   4.411e-01, 6.622e-01
  N Arrays:   2